In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
import matplotlib.pyplot as plt
import catboost as cat
import xgboost as xgb
import lightgbm as lgb
import shap
from datetime import timedelta, datetime
import warnings

warnings.filterwarnings("ignore")

## Input the 3 subsets of the regime (trainval, test24, test25)

train_val_df = pd.read_csv('./subsets_intervals/subset_r3.csv')           # here
test_df = pd.read_csv('./subsets_intervals/test_r3.csv')                  # here
forward_test_df = pd.read_csv('./subsets_intervals/forward_test_r3.csv')  # here
test_df['Time'] = pd.to_datetime(test_df['Time'])
forward_test_df['Time'] = pd.to_datetime(forward_test_df['Time'])
test_df = test_df[test_df['Time']>='2024-01-01']
train_val_df['Time'] = pd.to_datetime(train_val_df['Time'])
test_df = test_df.sort_values(by='Time', ascending=True)
forward_test_df = forward_test_df.sort_values(by='Time', ascending=True)
train_val_df.drop(columns='Regime',inplace=True)
test_df.drop(columns='Regime',inplace=True)
forward_test_df.drop(columns='Regime',inplace=True)

print(f"Train/Validation data shape: {train_val_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"Forward Test data shape: {forward_test_df.shape}")

Train/Validation data shape: (7885, 717)
Test data shape: (272, 717)
Forward Test data shape: (415, 717)


In [14]:
# Helper Functions
def calculate_stability_score(y_true, pred, class1_proportion):
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    if fp == 0:
        return 1e9 if tp > 0 else 0.0
    if class1_proportion == 0:
        return 0.0
    stability = (tp / fp) / class1_proportion
    return stability

In [15]:
def evaluate_model(y_true, y_prob, model_name, class1_proportion_val, thresholds=None):
    """
    Simplified evaluation function with built-in optimal threshold finding,
    with a filter to avoid selecting unrealistic 100% precision/recall based on very few true positives.
    Returns metrics dictionary.
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 100)
        
    thresholds = [t for t in thresholds if 0.001 < t < 0.999]
    results = []
    
    # Find optimal threshold
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        stability = calculate_stability_score(y_true, y_pred, class1_proportion_val)
        
        results.append({
            'threshold': threshold,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'stability': stability,
            'tn': tn, # Store TP count for later filtering
            'fp': fp,
            'fn': fn,
            'tp': tp,
            'score': (tp * 800) - (fp * 200) - (fn * 200),
            'score_1': (tp * 800) - (fp * 200),
        })

    if not results:  # Fallback if no thresholds
        # This part remains as a safety fallback
        threshold = 0.5
        y_pred = (y_prob >= threshold).astype(int)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        stability = calculate_stability_score(y_true, y_pred, class1_proportion_val)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    else:
        results_df = pd.DataFrame(results)
        misleading_options = (results_df['fp'] < 5) | (results_df['fn'] < 5) | (results_df['tn'] < 5)
        filtered_df = results_df[~(misleading_options)].copy()
        # Fallback if filtering removes all candidates (e.g., all predictions are extreme)
        if filtered_df.empty:
            print(f"Warning: All candidates filtered out for {model_name}. Reverting to original selection without extreme filtering.")
            filtered_df = results_df.copy() # Use original results if all are deemed extreme
        else:
            print(f"Extreme filtering is done, original results {results_df.shape[0]} row, filtered {filtered_df.shape[0]}")
        # Find best threshold from the filtered (or original) candidates
        # Prioritize max score_1, then max f1
        max_score_1 = filtered_df['score_1'].max()
        best_candidates = filtered_df[filtered_df['score_1'] == max_score_1]
        best_row = best_candidates.loc[best_candidates['f1'].idxmax()]
        
        threshold = best_row['threshold']
        precision = best_row['precision']
        recall = best_row['recall']
        f1 = best_row['f1']
        stability = best_row['stability']
        score = best_row['score']
        score_1 = best_row['score_1']
        
        # Re-calculate confusion matrix for the final selected optimal threshold
        y_pred = (y_prob >= threshold).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        
    return {
        'model': model_name,
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'stability': stability,
        'confusion_matrix': cm,
        'score': score,
        'score_1': score_1,
    }

In [16]:
def preprocess_data(train, valid=None):

    train, valid = train.copy(), valid.copy()
    X_train, y_train = train.drop(columns=['Time','class']), train['class']
    X_valid, y_valid = valid.drop(columns=['Time','class']), valid['class']

    class_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

    tmp_model_final_shap = cat.CatBoostClassifier(random_state=1, loss_function='Logloss', eval_metric='F1', early_stopping_rounds=10, scale_pos_weight=class_weight, verbose=0) # 
    tmp_model_final_shap.fit(X_train, y_train, cat_features=[len(X_train.columns)-1], eval_set=[(X_valid, y_valid)])
    explainer_final = shap.TreeExplainer(tmp_model_final_shap)
    shap_values_final = explainer_final.shap_values(X_train)
    features_shap_final = pd.DataFrame({'Feature':X_train.columns,'Importance':np.abs(shap_values_final).mean(axis=0)}).sort_values(by='Importance', ascending=False)
    top_features = list(features_shap_final.loc[features_shap_final['Importance']>0,'Feature'].values)

    X_train_selected = X_train[top_features]
    X_valid_selected = X_valid[top_features]

    # Feature Interactions (Multiplication on selected features)
    X_train_interacted = X_train_selected.copy()
    X_valid_interacted = X_valid_selected.copy()
    # features_for_interaction = [f for f in top_features if f != 'Regime']

    # for i in range(len(features_for_interaction)):
    #     for j in range(i + 1, len(features_for_interaction)):
    #         feature1 = features_for_interaction[i]
    #         feature2 = features_for_interaction[j]
    #         X_train_interacted[f'{feature1}_x_{feature2}'] = X_train_interacted[feature1] * X_train_interacted[feature2]
    #         X_valid_interacted[f'{feature1}_x_{feature2}'] = X_valid_interacted[feature1] * X_valid_interacted[feature2]

    return X_train_interacted, y_train, X_valid_interacted, y_valid, top_features

In [17]:
params_search_cat = {'depth': [4, 6, 8, 10],'learning_rate': [0.1, 0.05, 0.01, 0.005],'l2_leaf_reg': [1, 3, 5, 7, 9, 11],'bagging_temperature': [0, 0.5, 1, 3, 5],'subsample': [0.5, 0.7, 0.9, 1],'random_strength': [1e-9, 1e-6, 1e-3, 1e-1],'border_count': [32, 64, 128, 254]}
params_search_xgb = {'learning_rate': [0.1, 0.05, 0.01, 0.005],'max_depth': [3, 6, 10],'min_child_weight': [1, 3, 5],'subsample': [0.6, 0.8, 1.0],'colsample_bytree': [0.6, 0.8, 1.0],'gamma': [0, 0.1, 0.5],'reg_alpha': [0, 0.1, 1],'reg_lambda': [1, 5, 10]}
params_search_lgb = {'learning_rate': [0.1, 0.05, 0.01, 0.005],'max_depth': [3, 5, 7, 10, -1],'num_leaves': [15, 31, 63, 127],'min_child_samples': [10, 20, 30, 50],'subsample': [0.6, 0.8, 1.0],'colsample_bytree': [0.6, 0.8, 1.0],'reg_alpha': [0, 0.01, 0.1, 1],'reg_lambda': [0, 0.01, 0.1, 1],}

In [18]:
def train_base_model(X_train, y_train, X_valid=None, y_valid=None, model_name='CatBoost'):
    
    class_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
    feature_scaler = None
    print(f'Hyperparameter tuning, base model: {model_name}, {datetime.now()}')
    if model_name == 'CatBoost':
        model = cat.CatBoostClassifier(random_state=1, scale_pos_weight=class_weight, loss_function='Logloss',  
                                       eval_metric='F1', early_stopping_rounds=50, iterations=1000, verbose=0)
        def cat_features_regime(cols):
            # This function is dedicated for catboost model.fit() to accomodate for Regime column as categorical feature
            if 'Regime' in cols:
                return ['Regime']
            return None
        # search = RandomizedSearchCV(estimator=model, param_distributions=params_search_cat, n_iter=10, cv=TimeSeriesSplit(n_splits=3), random_state=1, scoring='f1', verbose=0, n_jobs=-1)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], cat_features=cat_features_regime(list(X_train.columns)))
    elif model_name == 'XGBoost':
        model = xgb.XGBClassifier(objective='binary:logistic', scale_pos_weight=class_weight, random_state=1, n_estimators=1000, early_stopping_rounds=10, enable_categorical=True, tree_method='hist', verbosity=0)
        # search = RandomizedSearchCV(estimator=model, param_distributions=params_search_xgb, n_iter=10, cv=TimeSeriesSplit(n_splits=3), random_state=1, scoring='f1', verbose=0, n_jobs=-1)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    elif model_name == 'LogReg':
        feature_scaler = StandardScaler()
        x_tr_scaled = feature_scaler.fit_transform(X_train)
        x_va_scaled = feature_scaler.transform(X_valid)
        model = LogisticRegression(max_iter=1000, random_state=1, solver='liblinear', class_weight={0: 1, 1: class_weight}, C=0.01, penalty='l1')
        model.fit(x_tr_scaled, y_train)
    elif model_name == 'LGBM':
        model = lgb.LGBMClassifier(class_weight='balanced', objective='binary', random_state=1, learning_rate= 0.01, verbosity=-1, n_estimators=1000)
        # search = RandomizedSearchCV(estimator=model, param_distributions=params_search_lgb, n_iter=10, cv=TimeSeriesSplit(n_splits=3), random_state=1, scoring='f1', verbose=0, n_jobs=-1)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_metric="binary_logloss", callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)])
    
    # if model_name != 'LogReg':
    #     model = search.best_estimator_

    # calibrated_model = CalibratedClassifierCV(estimator=model, method='isotonic', cv='prefit')
    

    print(f'Base model {model_name} is completed, {datetime.now()}')
    try:
        if model_name == 'LogReg':
            # calibrated_model.fit(x_va_scaled, y_valid)
            # prob = calibrated_model.predict_proba(x_va_scaled)[:, 1]
            prob = model.predict_proba(x_va_scaled)[:, 1]
        else:
            # calibrated_model.fit(X_valid, y_valid)
            # prob = calibrated_model.predict_proba(X_valid)[:, 1]
            prob = model.predict_proba(X_valid)[:, 1]
    except Exception as e:
        print(f"{model} failed: {e}. Filling with zeros.")
        prob = np.zeros(len(y_valid))

    return prob, model, feature_scaler

In [19]:
def preprocess_test_data(test_df, interval='3m'):
    
    assert interval in ["3m", "6m", "1y"]
    start_date = "2024-01-01"

    if interval == "1y":
        end_date = "2024-12-31"
    elif interval == "6m":
        end_date = "2024-06-30"
    elif interval == "3m":
        end_date = "2024-03-31"

    test_df = test_df[(test_df["Time"] >= start_date) & (test_df["Time"] <= end_date)]

    x_test = test_df.drop(columns=["Time", "class"])
    y_test = test_df["class"]

    x_test_selected = x_test[top_features] # align top features with training set
    # applying features interaction on the selected ones
    x_test_interacted = x_test_selected.copy()
    # features_for_interaction = [f for f in top_features if f != 'Regime'] # exclude Regime
    # for i in range(len(features_for_interaction)):
    #         for j in range(i + 1, len(features_for_interaction)):
    #             feature1 = features_for_interaction[i]
    #             feature2 = features_for_interaction[j]
    #             x_test_interacted[f'{feature1}_x_{feature2}'] = x_test_interacted[feature1] * x_test_interacted[feature2]
    assert list(X_train_interacted.columns) == list(x_test_interacted.columns)

    return x_test_interacted, y_test

In [20]:
def preprocess_forward_test_data(test_df, interval='3m'):
    
    assert interval in ["3m", "6m", "1y"]
    start_date = test_df['Time'].min()

    if interval == "1y":
        end_date = start_date + timedelta(days=365)
    elif interval == "6m":
        end_date = start_date + timedelta(days=180)
    elif interval == "3m":
        end_date = start_date + timedelta(days=90)

    test_df = test_df[(test_df["Time"] >= start_date) & (test_df["Time"] <= end_date)]

    x_test = test_df.drop(columns=["Time", "class"])
    y_test = test_df["class"]

    x_test_selected = x_test[top_features] # align top features with training set
    # applying features interaction on the selected ones
    x_test_interacted = x_test_selected.copy()
    # features_for_interaction = [f for f in top_features if f != 'Regime'] # exclude Regime
    # for i in range(len(features_for_interaction)):
    #         for j in range(i + 1, len(features_for_interaction)):
    #             feature1 = features_for_interaction[i]
    #             feature2 = features_for_interaction[j]
    #             x_test_interacted[f'{feature1}_x_{feature2}'] = x_test_interacted[feature1] * x_test_interacted[feature2]
    assert list(X_train_interacted.columns) == list(x_test_interacted.columns)

    return x_test_interacted, y_test

## Train base models on the full data (or partial)

In [ ]:
# df = train_val_df
# train_val_df = df[(df['Time']>='2008-01-01') & (df['Time']<='2015-05-22')]
# test_df = df[(df['Time']>='2015-05-23') & (df['Time']<='2016-01-01')]
# forward_test_df = df[df['Time']>='2016-01-01']

In [21]:
tr_start_date = datetime.strptime("2008-01-01", "%Y-%m-%d")
tr_end_date = datetime.strptime("2015-12-31", "%Y-%m-%d")
va_start_date = tr_end_date + timedelta(days=1)
va_end_date = train_val_df["Time"].max()
# va_end_date = datetime.strptime("2023-10-03", "%Y-%m-%d")
training_data = train_val_df[(train_val_df["Time"] >= tr_start_date) & (train_val_df["Time"] <= tr_end_date)]
validation_data = train_val_df[(train_val_df["Time"] >= va_start_date) & (train_val_df["Time"] <= va_end_date)]
print(f"Training Period: {tr_start_date.strftime('%Y-%m-%d')} - {tr_end_date.strftime('%Y-%m-%d')} | Validation Period: {va_start_date.strftime('%Y-%m-%d')} - {va_end_date.strftime('%Y-%m-%d')} \n")
print(f'Train sample size: {training_data.shape[0]}, Valid sample size: {validation_data.shape[0]}')
X_train_interacted, y_train, X_valid_interacted, y_valid, top_features = preprocess_data(training_data, validation_data)
lr_prob, lr_model, feature_scaler = train_base_model(X_train_interacted, y_train, X_valid_interacted, y_valid, model_name='LogReg')
###################################### Check valid metrics #########################################
class1_proportion = y_valid.value_counts(normalize=True)[1]
lr_metrics = evaluate_model(y_valid, lr_prob, "Validation", class1_proportion)
validation_data_metrics = pd.DataFrame([lr_metrics])
validation_data_metrics.round(5)

Training Period: 2008-01-01 - 2015-12-31 | Validation Period: 2016-01-01 - 2023-11-13 

Train sample size: 4304, Valid sample size: 3555
Hyperparameter tuning, base model: LogReg, 2025-08-23 16:38:47.808032
Base model LogReg is completed, 2025-08-23 16:38:47.819023
Extreme filtering is done, original results 100 row, filtered 13


,model,threshold,precision,recall,f1,stability,confusion_matrix,score,score_1
0,Validation,0.53465,0.25,0.00462,0.00906,1.82308,"[[2896, 9], [647, 3]]",-128800.0,600.0


## Testing on 2024 data

In [23]:
x_test_interacted, y_test = preprocess_test_data(test_df=test_df, interval='1y')
class1_proportion_test = y_valid.value_counts(normalize=True)[1]

lr_pred_test = lr_model.predict_proba(feature_scaler.transform(x_test_interacted))[:, 1]

# Rank-based threshold using 2024 data
lr_threshold = np.percentile(lr_pred_test, 75)

lr_metrics_test = evaluate_model(y_test, lr_pred_test, '', class1_proportion_test, [lr_threshold])

pd.DataFrame([lr_metrics_test])

Extreme filtering is done, original results 1 row, filtered 1


,model,threshold,precision,recall,f1,stability,confusion_matrix,score,score_1
0,,0.533482,0.411765,0.358974,0.383562,3.828462,"[[154, 40], [50, 28]]",4400.0,14400.0


## Forward Test Data 2025

In [25]:
x_test_interacted_2025, y_test_2025 = preprocess_forward_test_data(test_df=forward_test_df, interval='6m')
class1_proportion_test_2025 = y_valid.value_counts(normalize=True)[1]
lr_pred_test_2025 = lr_model.predict_proba(feature_scaler.transform(x_test_interacted_2025))[:, 1]

# Rank-based threshold using 2025 data
lr_threshold_2025 = np.percentile(lr_pred_test_2025, 75)

lr_metrics_test_2025 = evaluate_model(y_test_2025, lr_pred_test_2025, 'LogReg', class1_proportion_test_2025, [lr_threshold_2025])

pd.DataFrame([lr_metrics_test_2025])

Extreme filtering is done, original results 1 row, filtered 1


,model,threshold,precision,recall,f1,stability,confusion_matrix,score,score_1
0,LogReg,0.524988,0.384615,0.412371,0.39801,3.418269,"[[254, 64], [57, 40]]",7800.0,19200.0
